In [1]:
!nvidia-smi

# Install ultralytics
!pip install ultralytics==8.3.0

# Completely disable wandb so it doesn't break Drive paths
!pip uninstall -y wandb

import os
os.environ["WANDB_DISABLED"] = "true"

from ultralytics import YOLO
import yaml, os


Tue Nov 25 05:14:20 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# 👇 CHANGE THIS to your real zip path in Drive
# Path to your ZIP file on Drive
ZIP_PATH = "/content/drive/MyDrive/YOLOv8_Training/PPE.zip"

# Folder where you want to unzip the dataset
UNZIP_DIR = "/content/drive/MyDrive/PPE_Dataset_new"

# Create the folder if it doesn't exist
os.makedirs(UNZIP_DIR, exist_ok=True)

print("Using ZIP:", ZIP_PATH)
print("Unzipping into:", UNZIP_DIR)

# Unzip directly into your target folder on Drive
!unzip -q "$ZIP_PATH" -d "$UNZIP_DIR"

# Show the extracted files
!ls -R "$UNZIP_DIR"



Streaming output truncated to the last 5000 lines.
thumb0945_jpg.rf.11f7de8d019a87862705203bee777be6.txt
thumb0945_jpg.rf.5d4e530fe64a54e4f77177c32ca26269.txt
thumb0945_jpg.rf.725931483ca1404cdb7b829c85414871.txt
thumb1718_jpg.rf.31ca37e285710abe899691be92969679.txt
thumb1718_jpg.rf.a42cf32d1ff91d69752011a2cbdf03f8.txt
thumb1718_jpg.rf.c2b4fd36b5d339f7c7b9d7a5c0795a04.txt
thumb2434_jpg.rf.ea6c70acba018d6132299e5c9790c6c4.txt
thumb2435_jpg.rf.3db0481ea16639d7cba28bd22767d858.txt
thumb2435_jpg.rf.fef7ffc72572e6d305b3752550ad56b1.txt
thumb2443_jpg.rf.2c02bf1a1277fc70c2924ffb28a2854f.txt
thumb2443_jpg.rf.f0b1c1e8166446fa5d22e8faa164de8f.txt
thumb2444_jpg.rf.2e212b3cb5114d2d58f5142fd9f634db.txt
TMEE4781_JPEG.rf.6de9be429e8526e89af989b89b48df24.txt
TNPDu_78_jpg.rf.b31a9cbed0037243fd2898aaf0094ded.txt
TNPDu_78_jpg.rf.cf140a34e0c9d1faa9de525868c86940.txt
TNPDu_78_jpg.rf.ffafeb7e57e04803eeeecb09c2968024.txt
tools-floor-young-plumber-man-74304426_jpg.rf.1361f81a6f72dcf67a3ee31c9c19bc0b.txt
tools

In [4]:
import yaml

data = {
    "path": "/content/drive/MyDrive/PPE_Dataset_new",
    "train": "train/images",
    "val": "valid/images",      # change to 'train/images' if valid empty
    "test": "test/images",
    "nc": 6,
    "names": ['Gloves', 'Vest', 'goggles', 'helmet', 'mask', 'safety_shoe']
}

with open("/content/drive/MyDrive/PPE_Dataset_new/data.yaml", "w") as f:
    yaml.dump(data, f, default_flow_style=False)

print(open("/content/drive/MyDrive/PPE_Dataset_new/data.yaml").read())


names:
- Gloves
- Vest
- goggles
- helmet
- mask
- safety_shoe
nc: 6
path: /content/drive/MyDrive/PPE_Dataset_new
test: test/images
train: train/images
val: valid/images



In [5]:
!pip uninstall -y wandb

import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"


In [6]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # or yolov8s.pt if you prefer

results = model.train(
    data="/content/drive/MyDrive/PPE_Dataset_new/data.yaml",
    imgsz=640,
    epochs=50,
    batch=32,                 # reduce if OOM
    workers=4,
    device=0,                 # T4 GPU
    project="jetson_yolo_runs",
    name="jetson_detection",
    exist_ok=True
)

print("Training done.")




100%|██████████| 6.25M/6.25M [00:00<00:00, 100MB/s]


New https://pypi.org/project/ultralytics/8.3.231 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/drive/MyDrive/PPE_Dataset_new/data.yaml, epochs=50, time=None, patience=100, batch=32, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=4, project=jetson_yolo_runs, name=jetson_detection, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=Fa

100%|██████████| 755k/755k [00:00<00:00, 21.4MB/s]


Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

train: Scanning /content/drive/MyDrive/PPE_Dataset_new/train/labels... 8774 images, 2632 backgrounds, 0 corrupt: 100%|██████████| 8774/8774 [02:37<00:00, 55.85it/s] 


...

      50/50       4.6G     0.8497     0.5658      1.061          4        640: 100%|██████████| 275/275 [02:53<00:00,  1.58it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 33/33 [00:23<00:00,  1.39it/s]

                   all       2070       2504       0.81      0.747      0.806      0.535



50 epochs completed in 2.930 hours.
Optimizer stripped from jetson_yolo_runs/jetson_detection/weights/last.pt, 5.6MB
Optimizer stripped from jetson_yolo_runs/jetson_detection/weights/best.pt, 5.6MB

Validating jetson_yolo_runs/jetson_detection/weights/best.pt...
Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 186 layers, 2,685,538 parameters, 0 gradients, 6.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 33/33 [00:26<00:00,  1.27it/s]


                   all       2070       2504      0.809      0.745      0.806      0.536
                Gloves        297        558      0.773      0.593       0.69       0.52
                  Vest        481        842      0.835      0.893      0.907      0.747
               goggles        255        298      0.868      0.852      0.903      0.471
                helmet        169        210      0.711      0.905      0.899      0.676
                  mask        131        408      0.911      0.708        0.8      0.455
           safety_shoe        102        188      0.759       0.52      0.639      0.344
Speed: 0.2ms preprocess, 2.2ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to jetson_yolo_runs/jetson_detection
Training done.
